# ಪಾಠ 07 - ಯೋಜನಾ ವಿನ್ಯಾಸ ಮಾದರಿ

ಈ ನೋಟ್‌ಬುಕ್ ಮೈಕ್ರೋಸಾಫ್ಟ್ ಏಜೆಂಟ್ ಫ್ರೇಮ್ವರ್ಕ್ ಬಳಸಿ AI ಏಜೆಂಟ್‌ಗಳಿಗೆ **ಯೋಜನಾ ವಿನ್ಯಾಸ ಮಾದರಿ** ಅನ್ನು ಪ್ರದರ್ಶಿಸುತ್ತದೆ.
ನೀವು ಸಂಕೀರ್ಣ ಪ್ರಯಾಣ ವಿನಂತಿಗಳನ್ನು ರಚನಾತ್ಮಕ ಉಪಕಾರ್ಯಗಳಾಗಿ ವಿಭಜಿಸುವುದನ್ನು, ವಿಶೇಷತಜ್ಞ ಏಜೆಂಟ್‌ಗಳಿಗೆ ಇವುಗಳನ್ನು ನಿಯೋಜಿಸುವುದನ್ನು,
ಮತ್ತು ಫಲಿತಾಂಶ ಯೋಜನೆಯನ್ನು ಕಾರ್ಯಗತಗೊಳಿಸುವುದನ್ನು ಕಲಿಯಲಿದ್ದೀರಿ — ಎಲ್ಲವೂ ಪೈಡ್ಯಾಂಟಿಕ್ ಮಾದರಿಗಳಿಂದ ಚಾಲಿತ ರಚನಾತ್ಮಕ ಔಟ್‍ಪುಟ್ ಬಳಸಿಕೊಂಡು.


## ಸೆಟಪ್


In [ ]:
%pip install agent-framework azure-ai-projects azure-identity python-dotenv -q

In [ ]:
import logging
logging.getLogger("agent_framework.foundry").setLevel(logging.ERROR)

import os, asyncio
import dotenv
from typing import Annotated
from pydantic import BaseModel
from agent_framework import tool
from agent_framework.foundry import FoundryChatClient
from azure.identity import DefaultAzureCredential

dotenv.load_dotenv()

endpoint = os.getenv("AZURE_AI_PROJECT_ENDPOINT")
deployment_name = os.getenv("AZURE_AI_MODEL_DEPLOYMENT_NAME")

missing = [k for k, v in {
    "AZURE_AI_PROJECT_ENDPOINT": endpoint,
    "AZURE_AI_MODEL_DEPLOYMENT_NAME": deployment_name
}.items() if not v]

if missing:
    raise ValueError(
        f"Missing required environment variables: {', '.join(missing)}. "
        "Please set them as environment variables (e.g., in your .env file or shell environment)."
    )

In [ ]:
# Create the Microsoft Foundry client
client = FoundryChatClient(
    project_endpoint=endpoint,
    model=deployment_name,
    credential=DefaultAzureCredential()
)

## Task Decomposition

Task decomposition is the core of the planning design pattern. Instead of asking a single agent to
handle a complex request end-to-end, we break the problem into smaller, well-defined **subtasks**.
Each subtask is assigned to a specialist agent (e.g., flights, hotels, activities) with clear
priorities and dependency ordering.

This approach provides several benefits:
- **Clarity**: each subtask has a single responsibility.
- **Parallelism**: independent subtasks can run concurrently.
- **Reliability**: failures are isolated to individual subtasks.
- **Budget tracking**: costs are estimated per subtask and rolled up.


In [ ]:
class TravelSubTask(BaseModel):
    task_id: int
    description: str
    assigned_agent: str  # "flight_agent", "hotel_agent", "activity_agent"
    priority: str  # "high", "medium", "low"
    dependencies: list[int] = []


class TravelPlan(BaseModel):
    destination: str
    trip_duration_days: int
    subtasks: list[TravelSubTask]
    total_estimated_budget_usd: int
    notes: str

## ರಚನೆಯ ಆದೇಶ_output ಒದಗಿಸುವ ಯೋಜನಾ ಏಜೆಂಟ್ ರಚನೆ

ಯೋಜನಾ ಏಜೆಂಟ್ **ಮುಖ್ಯಸ್ಥರ ಸಂಯೋಜಕ** ಆಗಿ ಕಾರ್ಯನಿರ್ವಹಿಸುತ್ತದೆ. ಉನ್ನತ ಮಟ್ಟದ ಪ್ರವಾಸ ವಿನಂತಿಯನ್ನು ಪಡೆಯಲು,
ಅದು ರಚನೆಯ `TravelPlan` ಅನ್ನು ರಚಿಸುತ್ತದೆ — ವಿನಂತಿಯನ್ನು ಉಪಕಾರ್ಯಗಳಾಗಿ ವಿಭಜಿಸಿ, ಆದ್ಯತೆಗಳನ್ನು ನಿಗದಿ ಮಾಡಿ,
ಮತ್ತು ನಿಮಿಷಣೆಯನ್ನು ಗುರುತಿಸಿ, jotta a concierge ಅಥವಾ ಕಾರ್ಯಗತമಾಡುವ ಹಂತವು ಕೆಲಸವನ್ನು ನೆರವೇರಿಸಬಹುದು.


In [ ]:
planning_agent = client.as_agent(
    name="TravelPlanner",
    instructions="""You are a travel planning agent. When given a travel request:
1. Break it into specific subtasks (flights, hotels, activities, logistics)
2. Assign each subtask to the appropriate specialist agent
3. Set priorities and identify dependencies between tasks
4. Estimate the total budget""",
)

result = await planning_agent.run(
    "Plan a 7-day trip to Paris for a couple interested in art, cuisine, and history. Budget around $5000.",
    options={"response_format": TravelPlan}
)
if result:
    plan = result.value
    print(f"Destination: {plan.destination}")
    print(f"Duration: {plan.trip_duration_days} days")
    print(f"Budget: ${plan.total_estimated_budget_usd}")
    print(f"\nSubtasks:")
    for task in plan.subtasks:
        print(f"  [{task.priority}] {task.task_id}. {task.description} → {task.assigned_agent}")

## ನಿಪುಣ ಸಾಧನಗಳೊಂದಿಗೆ ಯೋಜನೆಯನ್ನು ಕಾರ್ಯಗತಗೊಳಿಸುವುದು

ಮುಂದಣ ಡೆಸ್ಕ್ ಪ್ರತಿನಿಧಿ ರಚಿಸಲಾದ ಸೂಚಿತ ಯೋಜನೆಯನ್ನು ಒದಗಿಸಿದ ಮೇಲೆ, **ಕಾನ್ಸಿಯರ್ಜ್ 에ಜೆಂಟ್** ಅದನ್ನು ನಿರ್ವಹಿಸುತ್ತದೆ.
ಪ್ರತಿ ನಿಪುಣ ಸಾಧನವು ಒಂದು ಉಪಕಾರ್ಯ ವಿಭಾಗವನ್ನು ನಿರ್ವಹಿಸುತ್ತದೆ (ಏರ್‌ಲೈನ್, ಹೋಟೆಲ್‌ಗಳು, ಚಟುವಟಿಕೆಗಳು). ಕಾನ್ಸಿಯರ್ಜ್
ಆ ಯೋಜನೆಯ ಉಪಕಾರ್ಯಗಳನ್ನು ಅವಲಂಬನೆ ಕ್ರಮದಲ್ಲಿ ಕ್ರಮೇಕ্রমವಾಗಿ ಪರಿಗಣಿಸಿ ಪ್ರತಿ ಒಂದನ್ನು
ಸಂಬಂಧಿಸಿದ ಸಾಧನಕ್ಕೆ ಕಳುಹಿಸುತ್ತದೆ.


In [ ]:
@tool
def book_flight(
    destination: Annotated[str, "The destination city"],
    departure_date: Annotated[str, "Departure date (YYYY-MM-DD)"],
    return_date: Annotated[str, "Return date (YYYY-MM-DD)"],
) -> str:
    """Search and book flights for the trip."""
    return f"Flight booked to {destination}: {departure_date} → {return_date}, confirmation #FLT-{hash(destination) % 10000:04d}"


@tool
def reserve_hotel(
    city: Annotated[str, "The city for the hotel"],
    check_in: Annotated[str, "Check-in date (YYYY-MM-DD)"],
    check_out: Annotated[str, "Check-out date (YYYY-MM-DD)"],
    guests: Annotated[int, "Number of guests"],
) -> str:
    """Reserve a hotel room in the destination city."""
    return f"Hotel reserved in {city}: {check_in} to {check_out} for {guests} guests, confirmation #HTL-{hash(city) % 10000:04d}"


@tool
def book_activity(
    activity_name: Annotated[str, "Name of the activity or tour"],
    date: Annotated[str, "Date of the activity (YYYY-MM-DD)"],
    participants: Annotated[int, "Number of participants"],
) -> str:
    """Book a tour, museum visit, or other activity."""
    return f"Activity booked: {activity_name} on {date} for {participants} people, confirmation #ACT-{hash(activity_name) % 10000:04d}"


# Concierge agent that executes the plan using specialist tools
concierge_agent = client.as_agent(
    name="Concierge",
    instructions="""You are a travel concierge executing a structured travel plan.
Use the available tools to fulfil each subtask. Work through the subtasks in order,
respecting dependencies. Summarise the results when finished.""",
    tools=[book_flight, reserve_hotel, book_activity],
)

# Build a prompt from the plan produced above
if result.value:
    subtask_lines = "\n".join(
        f"- [{t.priority}] {t.task_id}. {t.description} (agent: {t.assigned_agent}, deps: {t.dependencies})"
        for t in plan.subtasks
    )
    execution_prompt = (
        f"Execute the following travel plan for {plan.destination} "
        f"({plan.trip_duration_days} days, ${plan.total_estimated_budget_usd} budget):\n"
        f"{subtask_lines}"
    )

    exec_response = await concierge_agent.run(execution_prompt)
    print(exec_response)

## ಸಾರಾಂಶ

ಈ ಪಾಠದಲ್ಲಿ ನೀವು AI ಏಜೆಂಟ್‌ಗಳಿಗಾಗಿ **ಯೋಜನೆ ವಿನ್ಯಾಸ ಮಾದರಿ** ಕುರಿತು ತಿಳಿದುಕೊಳ್ಳಲಾಯಿತು:

1. **ಕಾರ್ಯ ವಿಭಜನೆ** — ಫ್ರಂಟ್ ಡೆಸ್ಕ್ ಯೋಜನಾ ಏಜೆಂಟ್ ಒಂದು ಸಂಕೀರ್ಣ ಪ್ರಯಾಣ ವಿನಂತಿಯನ್ನು
   ಪೈಡ್ಯಾಂಟಿಕ್ ಮಾದರಿಗಳನ್ನು ಬಳಸಿಕೊಂಡು ಸಂರಚಿತ ಉಪಕಾರ್ಯಗಳಿಗೆ ವಿಭಜಿಸಿ, ಪ್ರತಿ ಉಪಕಾರ್ಯವನ್ನು ಅನುವೇಶಕ ಟ್ರಿಗರ್‌ಗಳೊಂದಿಗೆ
   ತಂಡದ ವಿಶೇಷಜ್ಞ ಏಜೆಂಟ್‌ಗೆ ಆದ್ಯತೆ ಮತ್ತು ಅವಲಂಬನೆಗಳನ್ನು ನಿಗದಿಪಡಿಸುತ್ತದೆ.
2. **ಸಂರಚಿತ_OUTPUT** — `response_format` ಅನ್ನು ಪಾಸು ಮಾಡುವ ಮೂಲಕ ಏಜೆಂಟ್ ಸಹಿತ
   `TravelPlan` ವಸ್ತು ಮಾನ್ಯತೆಯೊಂದಿಗೆ ಹೊರಡಿಸುತ್ತದೆ, ಮುಕ್ತರೂಪದ ಪಠ್ಯಕ್ಕೆ ಬದಲಿ, ಬಹಳ ನಂಬಬಹುದಾದ ದಕ್ಷಿಣಪ್ರಸರಣೆಯನ್ನು ಮಾಡುತ್ತದೆ.
3. **ಯೋಜನೆ ಕಾರ್ಯಗತಗೊಳಿಸುವಿಕೆ** — ಕಂಸಿಯರ್ಜ್ ಏಜೆಂಟ್ ಉಪಕಾರ್ಯಗಳನ್ನು ವಿಶೇಷಜ್ಞ ಉಪಕರಣಗಳ (
   `book_flight`, `reserve_hotel`, `book_activity`) ಉಪಯೋಗಿಸಿ ಪ್ಲ್ಯಾನ್ ಅನ್ನು ಅನುಸರಿಸಿ ಫಲಿತಾಂಶಗಳನ್ನು ವರದಿ ಮಾಡುತ್ತದೆ.

ಈ ಮಾದರಿ *ಏನು ಮಾಡಬೇಕೆಂದು* (ಯೋಜನೆಗೊಳಿಸುವಿಕೆ) ಮತ್ತು *ಇದನ್ನು ಹೇಗೆ ಮಾಡುವುದು* (ಕಾರ್ಯಗತಗೊಳಿಸುವಿಕೆ) ಅನ್ನು ವಿಭಜಿಸುತ್ತದೆ,
ಏಜೆಂಟ್‌ಗಳನ್ನು ಹೆಚ್ಚು ಘಟಕ, ಪರೀಕ್ಷಿಸಲು ಸುಲಭ ಮತ್ತು ವಿಸ್ತರಿಸಲು ಅನುಕೂಲಕರ ಮಾಡುತ್ತದೆ.


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**ಅಸ್ವೀಕಾರ**:
ಈ ದಸ್ತಾವೇಜು AI ಅನುವಾದ ಸೇವೆ [Co-op Translator](https://github.com/Azure/co-op-translator) ಬಳಸಿ ಅನುವಾದಿಸಲಾಗಿದೆ. ನಾವು ನಿಖರತೆಯನ್ನು ಸಾಧಿಸಲು ಪ್ರಯತ್ನಿಸುತ್ತಿದ್ದರೂ, ದಯವಿಟ್ಟು ಗಮನಿಸಿ, ಸ್ವಯಂಚಾಲಿತ ಅನುವಾದಗಳಲ್ಲಿ ದೋಷಗಳು ಅಥವಾ ಅಸಡ್ಡೆಗಳು ಇರಬಹುದು. ಮೂಲ ಭಾಷೆಯಲ್ಲಿರುವ ಮೂಲ ದಸ್ತಾವೇಜು ಪ್ರಾಮಾಣಿಕ ಮೂಲವೆಂದು ಪರಿಗಣಿಸಬೇಕು. ಪ್ರಮುಖ ಮಾಹಿತಿಗಾಗಿ, ವೃತ್ತಿಪರ ಮಾನವ ಅನುವಾದವನ್ನು ಶಿಫಾರಸು ಮಾಡಲಾಗುತ್ತದೆ. ಈ ಅನುವಾದವನ್ನು ಬಳಸುವ ಮೂಲಕ ಉಂಟಾಗುವ ಯಾವುದೇ ತಪ್ಪು ಅರ್ಥಗಳ ಅಥವಾ ತಪ್ಪು ವ್ಯಾಖ್ಯಾನಗಳ ಬಗ್ಗೆ ನಾವು ಹೊಣೆಗಾರರಲ್ಲ.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
